In [1]:
# HAIKUTHON - rerank speed test v2: STRUCTURAL speed-ups (run in parallel with rerank_speed_test v1)
# v1 tests image size / prompt / int8 / K / gating. v2 tests things v1 does not:
#   (a) EARLY EXIT: score yes/no from an intermediate LLM layer (12/16/20 of 28) -> faster AND smaller zip
#       (layers after the exit are simply not shipped). All exits are scored from ONE forward pass via hooks.
#   (b) even smaller VLM images: 256px (64 visual tokens) next to 320px (100 tokens)
#   (c) LISTWISE: one prompt per query with all K candidate images, "which image matches? answer 1-4"
#       (Qwen3-VL-2B-Instruct) -> the query text is read once instead of K times.
# Same team conditions as v1: HF dataset, RandomState(42) 500 val, cuda:0, fp16, /kaggle/working, resume.

In [2]:
# %% [1] install (pip skips anything already satisfied). Qwen3-VL needs transformers >= 4.57.
!pip install -q "transformers>=4.57.0"

In [3]:
# %% [2] settings
STAGE1 = "google/siglip2-large-patch16-384"
TAG = "siglip2L"                       # reuses zeroshot_eval_siglip2L embeddings if present in /kaggle/working
RERANKER = "Qwen/Qwen3-VL-Reranker-2B"
LISTWISE_MODEL = "Qwen/Qwen3-VL-2B-Instruct"
OUT, DEVICE = "/kaggle/working", "cuda:0"
KMAX = 4
EXITS = [12, 16, 20, 28]               # LLM layers to read the answer from (28 = full model)
RES_LIST = [320, 256]                  # VLM image sizes (multiples of 32): 100 / 64 visual tokens
BATCH, BATCH_Q = 16, 4                 # pointwise pairs per forward / listwise queries per forward
ALPHAS = [1, 2, 4, None]
T4_TO_L4 = 1.8                         # ponytail: rough fp16 FLOPs ratio, verify with one real submission
SERVER_CPUS = 4
LIMIT_MIN = 13.5
N_Q, N_IMG = 8000, 16000

import os, json, time, re
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
os.makedirs(OUT, exist_ok=True)

In [4]:
# %% [3] dataset + team split
from datasets import load_dataset
ds = load_dataset("KimWooJin/26-haikuthon-train", split="train")
CAP = next(c for c in ["prompts", "prompt", "text", "caption"] if c in ds.column_names)
idx = np.random.RandomState(42).permutation(len(ds))
val_idx = sorted(idx[:500].tolist())
json.dump(val_idx, open(f"{OUT}/val_idx.json", "w"))
FIRST10 = val_idx[:10]
print("first 10 val_idx:", FIRST10)
VORIG = [ds[i]["image"].convert("RGB") for i in val_idx]
RAW = [ds[i][CAP].strip() for i in val_idx]
LOW = [t.lower() for t in RAW]

README.md:   0%|          | 0.00/4.49k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/4001 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/4001 [00:00<?, ?it/s]

images/0006.png: reconstructing file:   0%|          |  0.00B /  855kB            

images/0011.png: reconstructing file:   0%|          |  0.00B /  589kB            

images/0004.png: reconstructing file:   0%|          |  0.00B /  893kB            

... (8,001 outputs left out)

images/3998.png: downloading bytes:           |  0.00B            

metadata.jsonl:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4000 [00:00<?, ? examples/s]

first 10 val_idx: [8, 12, 14, 17, 23, 26, 32, 33, 44, 45]


In [5]:
# %% [4] stage 1: SigLIP2-L zero-shot (identical to v1), reuse saved embeddings if present
from transformers import AutoModel, AutoProcessor
proc1 = AutoProcessor.from_pretrained(STAGE1)
s1 = AutoModel.from_pretrained(STAGE1, attn_implementation="sdpa").to(DEVICE).half().eval()
s1_gb = sum(p.numel() for p in s1.parameters()) * 2 / 1e9
trunc = float((np.array([len(x) for x in proc1.tokenizer(LOW)["input_ids"]]) > 64).mean())
img_path, spd_path = f"{OUT}/{TAG}_val_img_emb.npy", f"{OUT}/{TAG}_speed.json"
with torch.no_grad():
    if os.path.exists(img_path) and os.path.exists(spd_path):
        I = torch.from_numpy(np.load(img_path)); sp = json.load(open(spd_path)); print("reused", img_path)
    else:
        s1.vision_model(pixel_values=proc1(images=VORIG[:1], return_tensors="pt")["pixel_values"].to(DEVICE).half())
        cpu_t = gpu_t = 0.0; E = []
        for s in range(0, 500, 32):
            t0 = time.time(); pv = proc1(images=VORIG[s:s + 32], return_tensors="pt")["pixel_values"]; cpu_t += time.time() - t0
            torch.cuda.synchronize(DEVICE); t0 = time.time()
            E.append(F.normalize(s1.vision_model(pixel_values=pv.to(DEVICE).half()).pooler_output.float(), dim=-1).cpu())
            torch.cuda.synchronize(DEVICE); gpu_t += time.time() - t0
        I = torch.cat(E); np.save(img_path, I.numpy())
        sp = {"gpu_img_per_s": 500 / gpu_t, "e2e_img_per_s": 500 / (cpu_t + gpu_t)}; json.dump(sp, open(spd_path, "w"))
    torch.cuda.synchronize(DEVICE); t0 = time.time(); E = []
    for s in range(0, 500, 64):
        ids = proc1(text=LOW[s:s + 64], padding="max_length", max_length=64, truncation=True, return_tensors="pt")["input_ids"].to(DEVICE)
        E.append(F.normalize(s1.text_model(input_ids=ids).pooler_output.float(), dim=-1).cpu())
    torch.cuda.synchronize(DEVICE); txt_s_per = (time.time() - t0) / 500
    T = torch.cat(E); np.save(f"{OUT}/{TAG}_val_txt_emb.npy", T.numpy())
del s1; torch.cuda.empty_cache()
cpu_s_per = max(1 / sp["e2e_img_per_s"] - 1 / sp["gpu_img_per_s"], 0)
s1_min = (N_IMG / (sp["gpu_img_per_s"] * T4_TO_L4) + N_IMG * cpu_s_per / SERVER_CPUS + N_Q * txt_s_per / T4_TO_L4) / 60
sim = T @ I.T
rank1 = ((sim > sim.diag()[:, None]).sum(1) + 1).numpy()
topv, topi = sim.topk(KMAX, dim=1)
margin = (topv[:, 0] - topv[:, 1]).numpy()
print(f"stage-1 est. on L4: {s1_min:.1f} min | correct in top-2/3/4: " + " / ".join(f"{(rank1 <= k).mean():.3f}" for k in (2, 3, 4)))

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.53GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/792 [00:00<?, ?it/s]

stage-1 est. on L4: 3.9 min | correct in top-2/3/4: 0.950 / 0.994 / 0.996


In [6]:
# %% [5] (a)+(b) early exit: one full forward per pair, answer read at several layers through hooks
from transformers import AutoProcessor as AP, Qwen3VLForConditionalGeneration
JUDGE_SYS = 'Judge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".'
INSTR = "<Instruct>: Retrieve images that match the user's description exactly.<Query>:"
def conv(text, img):   # Qwen3-VL-Reranker format (scripts/qwen3_vl_reranker.py), approximated
    return [{"role": "system", "content": [{"type": "text", "text": JUDGE_SYS}]},
            {"role": "user", "content": [{"type": "text", "text": INSTR}, {"type": "text", "text": text},
                                         {"type": "text", "text": "\n<Document>:"}, {"type": "image", "image": img}]}]

proc2 = AP.from_pretrained(RERANKER); proc2.tokenizer.padding_side = "left"
vocab = proc2.tokenizer.get_vocab()
YES, NO = [vocab[w] for w in ["yes", "Yes"] if w in vocab], [vocab[w] for w in ["no", "No"] if w in vocab]
m = Qwen3VLForConditionalGeneration.from_pretrained(RERANKER, dtype=torch.float16, attn_implementation="sdpa").to(DEVICE).eval()
lm = m.model.language_model                      # Qwen3VLTextModel: .layers (28 decoder layers) + .norm
ALL_LAYERS = lm.layers
W = m.lm_head.weight[YES + NO].float()
def yes_no(h): lg = lm.norm(h).float() @ W.T; return torch.logsumexp(lg[:, :len(YES)], 1) - torch.logsumexp(lg[:, len(YES):], 1)

CAPT = {}
def make_hook(L):
    def hook(mod, inp, out): CAPT[L] = (out[0] if isinstance(out, tuple) else out)[:, -1]
    return hook
hooks = [ALL_LAYERS[L - 1].register_forward_hook(make_hook(L)) for L in EXITS]

pairs = [(a, b) for a in range(500) for b in range(KMAX)]
def batches(imgs, pp, bs):
    for s in range(0, len(pp), bs):
        ch = pp[s:s + bs]
        yield s, ch, proc2.apply_chat_template([conv(RAW[a], imgs[int(topi[a, b])]) for a, b in ch], tokenize=True,
                                               add_generation_prompt=True, return_dict=True, padding=True, return_tensors="pt").to(DEVICE)

RES = {}
with torch.no_grad():
    for res in RES_LIST:
        f = f"{OUT}/rr2_exit_r{res}.pt"
        if os.path.exists(f): RES.update(torch.load(f)); print("reused", f); continue
        imgs = [im.resize((res, res), Image.BICUBIC) for im in VORIG]
        S = {L: torch.zeros(len(pairs)) for L in EXITS}
        for s, ch, inp in batches(imgs, pairs, BATCH):
            m.model(**inp, use_cache=False)
            for L in EXITS: S[L][s:s + len(ch)] = yes_no(CAPT[L]).cpu()
        # timing: actually drop the layers after the exit (what model.py would ship)
        out = {}
        for L in EXITS:
            lm.layers = nn.ModuleList(ALL_LAYERS[:L])
            sub = pairs[:256]
            for _, _, inp in batches(imgs, sub[:BATCH], BATCH): m.model(**inp, use_cache=False)   # warm-up
            torch.cuda.synchronize(DEVICE); t0 = time.time()
            for _, _, inp in batches(imgs, sub, BATCH): m.model(**inp, use_cache=False)
            torch.cuda.synchronize(DEVICE); pps = len(sub) / (time.time() - t0)
            lm.layers = ALL_LAYERS
            gb = (sum(p.numel() for p in m.parameters()) - sum(p.numel() for p in ALL_LAYERS[L:].parameters())) * 2 / 1e9
            if torch.isnan(S[L]).any(): print(f"WARNING exit{L} r{res}: NaN scores")
            out[f"exit{L}_r{res}"] = {"R": S[L].view(500, KMAX), "pps": pps, "gb": gb, "per": "pair"}
            print(f"exit{L}_r{res}: {pps:.1f} pairs/s (T4), reranker fp16 {gb:.2f} GB")
        torch.save(out, f); RES.update(out)
for h in hooks: h.remove()
del m, lm, ALL_LAYERS, W; torch.cuda.empty_cache()

preprocessor_config.json:   0%|          | 0.00/628 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.29k [00:00<?, ?B/s]

reranker.jinja:   0%|          | 0.00/2.44k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.65k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.45k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

... (22 outputs left out)

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

exit20_r256: 24.1 pairs/s (T4), reranker fp16 3.45 GB


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

exit28_r256: 19.1 pairs/s (T4), reranker fp16 4.26 GB


In [7]:
# %% [6] (c) listwise: one prompt per query with K candidate images, logits of the digits "1".."K"
f = f"{OUT}/rr2_listwise_r320.pt"
if os.path.exists(f):
    RES.update(torch.load(f)); print("reused", f)
else:
    proc3 = AP.from_pretrained(LISTWISE_MODEL); proc3.tokenizer.padding_side = "left"
    DIG = [proc3.tokenizer.get_vocab()[str(j + 1)] for j in range(KMAX)]
    m3 = Qwen3VLForConditionalGeneration.from_pretrained(LISTWISE_MODEL, dtype=torch.float16, attn_implementation="sdpa").to(DEVICE).eval()
    W3 = m3.lm_head.weight[DIG].float()
    imgs = [im.resize((320, 320), Image.BICUBIC) for im in VORIG]
    def lconv(a):   # candidates in stage-1 order (note: possible position bias)
        c = [{"type": "text", "text": f"Description: {RAW[a]}\nThere are {KMAX} candidate images."}]
        for j in range(KMAX):
            c += [{"type": "text", "text": f"\nImage {j + 1}:"}, {"type": "image", "image": imgs[int(topi[a, j])]}]
        c += [{"type": "text", "text": f"\nWhich image matches the description exactly (every object, color, count and position)? "
                                       f"Answer with only the image number (1-{KMAX})."}]
        return [{"role": "user", "content": c}]
    def lrun(qs):
        R = torch.zeros(len(qs), KMAX)
        for s in range(0, len(qs), BATCH_Q):
            inp = proc3.apply_chat_template([lconv(a) for a in qs[s:s + BATCH_Q]], tokenize=True, add_generation_prompt=True,
                                            return_dict=True, padding=True, return_tensors="pt").to(DEVICE)
            R[s:s + BATCH_Q] = torch.log_softmax(m3.model(**inp, use_cache=False).last_hidden_state[:, -1].float() @ W3.T, 1).cpu()
        return R
    with torch.no_grad():
        lrun([0])                                                  # warm-up
        torch.cuda.synchronize(DEVICE); t0 = time.time()
        R = lrun(list(range(500)))
        torch.cuda.synchronize(DEVICE); qps = 500 / (time.time() - t0)
    gb = sum(p.numel() for p in m3.parameters()) * 2 / 1e9
    out = {"listwise_r320": {"R": R, "pps": qps, "gb": gb, "per": "query"}}
    torch.save(out, f); RES.update(out)
    print(f"listwise_r320: {qps:.1f} queries/s (T4) = {qps * KMAX:.1f} candidate images/s, fp16 {gb:.2f} GB")
    del m3; torch.cuda.empty_cache()

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (4 outputs left out)

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

listwise_r320: 4.7 queries/s (T4) = 18.7 candidate images/s, fp16 4.26 GB


In [8]:
# %% [7] evaluate every combination -> table (score vs estimated L4 time)
def metrics(r):
    r = np.asarray(r, float); r1 = (r == 1).astype(float); nd = np.where(r <= 5, 1 / np.log2(r + 1), 0)
    return r1.mean(), nd.mean(), ((r1 + nd) / 2).mean()
def zrow(x): return (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + 1e-6)
def ranks_after(R, K, gate, alpha):
    f = R[:, :K].clone() if alpha is None else zrow(topv[:, :K]) + alpha * zrow(R[:, :K])
    order = f.argsort(1, descending=True); new = rank1.copy()
    for a in np.where(gate & (rank1 <= K))[0]:
        new[a] = (topi[a, :K][order[a]] == a).nonzero()[0, 0].item() + 1
    return new

THR = [("all", np.inf)] + [(f"margin<{np.quantile(margin, qq):.4f}", float(np.quantile(margin, qq))) for qq in (0.75, 0.5, 0.25)]
r1_, nd_, sc_ = metrics(rank1)
rows = [dict(variant="stage-1 only", K=0, rerank="-", alpha="-", frac=0.0, R1=r1_, NDCG5=nd_, score=sc_, est_min=s1_min, gb=s1_gb)]
for name, d in RES.items():
    for K in ((KMAX,) if d["per"] == "query" else (2, 3, 4)):      # listwise was prompted with exactly KMAX images
        for tname, t in THR:
            gate = margin < t; best = None
            for al in ALPHAS:
                m_ = metrics(ranks_after(d["R"], K, gate, al))
                if best is None or m_[2] > best[1][2]: best = (al, m_)
            work = N_Q * gate.mean() * (K if d["per"] == "pair" else 1)
            rows.append(dict(variant=name, K=K, rerank=tname, alpha="only" if best[0] is None else best[0], frac=gate.mean(),
                             R1=best[1][0], NDCG5=best[1][1], score=best[1][2],
                             est_min=s1_min + work / (d["pps"] * T4_TO_L4) / 60, gb=s1_gb + d["gb"]))
tab = pd.DataFrame(rows).sort_values("score", ascending=False)
tab.to_csv(f"{OUT}/rerank_speed_table_v2.csv", index=False)
pd.set_option("display.width", 200)
fmt = {"frac": "{:.0%}".format, "R1": "{:.4f}".format, "NDCG5": "{:.4f}".format, "score": "{:.4f}".format,
       "est_min": "{:.1f}".format, "gb": "{:.2f}".format}
fits = tab[(tab.est_min <= LIMIT_MIN) & (tab.gb <= 5.0)]
print(f"=== fits in {LIMIT_MIN} min AND 5 GB fp16 (top 10) ==="); print(fits.head(10).to_string(index=False, formatters=fmt))
print("\n=== all (top 15) ==="); print(tab.head(15).to_string(index=False, formatters=fmt))
print("\n=== per variant: best score regardless of time ===")
print(tab.groupby("variant").head(1).to_string(index=False, formatters=fmt))
print("\nnote: alpha/thresholds chosen on the same 500 val queries (slightly optimistic); time = T4 / "
      f"{T4_TO_L4} + decode estimate; gb = fp16 (int8 storage would roughly halve the reranker part)")

=== fits in 13.5 min AND 5 GB fp16 (top 10) ===
    variant  K        rerank alpha frac     R1  NDCG5  score est_min   gb
exit16_r256  3 margin<0.1125  only  75% 0.8940 0.9559 0.9249    10.0 4.81
exit16_r256  3           all  only 100% 0.8940 0.9559 0.9249    12.0 4.81
exit16_r256  4 margin<0.1125  only  75% 0.8940 0.9553 0.9247    12.0 4.81
exit16_r320  3 margin<0.1125  only  75% 0.8920 0.9559 0.9240    11.9 4.81
exit16_r320  3 margin<0.0446  only  50% 0.8820 0.9517 0.9168     9.2 4.81
exit16_r320  4 margin<0.0446  only  50% 0.8820 0.9516 0.9168    11.0 4.81
exit16_r256  3 margin<0.0446  only  50% 0.8800 0.9502 0.9151     8.0 4.81
exit16_r256  4 margin<0.0446  only  50% 0.8800 0.9496 0.9148     9.3 4.81
exit16_r320  2 margin<0.1125     1  75% 0.8760 0.9456 0.9108     9.2 4.81
exit16_r320  2           all     1 100% 0.8760 0.9456 0.9108    11.0 4.81

=== all (top 15) ===
    variant  K        rerank alpha frac     R1  NDCG5  score est_min   gb
exit16_r256  3 margin<0.1125  only  75% 0.

In [9]:
# %% [8] team-share line for the best combination that fits
b = fits.iloc[0] if len(fits) else tab.iloc[0]
if not len(fits): print("nothing fits -> showing the best overall")
rr = "" if b.variant == "stage-1 only" else (f" + Qwen3-VL-2B-Instruct [listwise]" if b.variant.startswith("listwise")
                                            else f" + Qwen3-VL-Reranker-2B [{b.variant}]") + f" K={b.K}, {b.rerank}, alpha={b.alpha}"
print("\n[팀 공유용]")
print(f"siglip2-large-patch16-384{rr} / {FIRST10} / R@1 {b.R1:.4f} / NDCG@5 {b.NDCG5:.4f} / 최종 {b.score:.4f} / "
      f"16,000장 {b.est_min:.1f}분 (L4 추정, 재정렬 포함) / {b.gb:.2f} GB / 잘림 {trunc:.1%} (SigLIP >64 tok, VLM 0%)")


[팀 공유용]
siglip2-large-patch16-384 + Qwen3-VL-Reranker-2B [exit16_r256] K=3, margin<0.1125, alpha=only / [8, 12, 14, 17, 23, 26, 32, 33, 44, 45] / R@1 0.8940 / NDCG@5 0.9559 / 최종 0.9249 / 16,000장 10.0분 (L4 추정, 재정렬 포함) / 4.81 GB / 잘림 2.0% (SigLIP >64 tok, VLM 0%)
